## Proyecto MathIA
### Clasificación desbalanceada para el diagnóstico de diferencias cognitivas y de aprendizaje en la enseñanza de las matemáticas
#### Clasificación

Este programa realiza los siguientes pasos, simulando una **puesta en producción**:

FASE DE PREDICCIÓN / CLASIFICACIÓN:

- Carga el modelo, el codificador de etiquetas y la lista con la estructura de columnas de los ficheros serializados .pkl.
- Carga un conjunto de datos de nuevos estudiantes en formato .csv.
- Clasifica los nuevos estudiantes e imprime los resultados. 

**Detalles en:**

Ana F. Hernández, Andrej Franulic, Javier A. Salazar, José Palma, Fernando Jiménez (2025), Imbalanced Classification for Diagnosing Learning and Cognitive Differences in Mathematics Education, XI Conference - European Congress of Methodology (EAM2025), La Laguna, Santa Cruz de Tenerife, Spain

##### Autor:
Fernando Jiménez Barrionuevo

In [1]:
import numpy as np
import pandas as pd
import joblib  # Librería estándar cargar los artefactos del modelo

# =============================================================================
# FASE 2: PREDICCIÓN / CLASIFICACIÓN
# =============================================================================

# Cargar modelo, codificador de etiquetas y lista con la estructura de columnas usadas en el entrenamiento
clf_cargado = joblib.load('modelo.pkl')
le_cargado = joblib.load('label_encoder.pkl')
columnas_entrenamiento_cargadas = joblib.load('columnas_entrenamiento.pkl')

# Cargar fichero con los nuevos estudiantes
file_name_est = 'nuevos_estudiantes.csv'
df_prediccion = pd.read_csv(file_name_est)
print(f"Fichero de datos de nuevos estudiantes: {file_name_est}")

# Guardar los identificadores y los eliminarlos del dataframe
if "Identificador" in df_prediccion.columns:
    identificadores = df_prediccion["Identificador"].tolist()
    df_prediccion = df_prediccion.drop(columns=["Identificador"])
else:
    # Salvaguarda: si no viene la columna, crear identificadores por posición
    identificadores = [f"Estudiante_{i+1}" for i in range(len(df_prediccion))]

# Quitar la columna objetivo si el fichero para predicción la trae (p.e. un test set)
if df_prediccion.shape[1] > 17: 
    X_pred = df_prediccion.iloc[:, :-1]
else:
    X_pred = df_prediccion

# Transformar las variables categóricas del nuevo flujo
X_pred_encoded = pd.get_dummies(X_pred)

# Forzar a que tenga EXACTAMENTE las mismas columnas del entrenamiento.
# Reindex resuelve ambos problemas: descarta categorías nuevas desconocidas y añade con 0 las que falten.
X_pred_final = X_pred_encoded.reindex(columns=columnas_entrenamiento_cargadas, fill_value=0)

# Predicción numérica (retorna 0 o 1)
predicciones_num = clf_cargado.predict(X_pred_final)

# Invertir la codificación para recuperar las etiquetas de texto originales
predicciones_texto = le_cargado.inverse_transform(predicciones_num)

print("Predicciones de nuevos estudiantes:")
for ident, pred in zip(identificadores, predicciones_texto):
    print(f"ID: {ident} -> Clasificación: {pred}")

print("\nEjecución completada correctamente.")

Fichero de datos de nuevos estudiantes: nuevos_estudiantes.csv
Predicciones de nuevos estudiantes:
ID: 1 -> Clasificación: D1
ID: 2 -> Clasificación: D0

Ejecución completada correctamente.
